# A Python programmer's tour of PeTTa

Start with an atom you can inspect: `S.Parent(S.Tom, S.Bob)` builds `(Parent Tom Bob)`. `S` creates symbols, `V` creates variables, and calling a symbol builds an expression. These operations stay in Python. `MeTTa().new_space()` starts an isolated engine-backed space for the examples below.

In [1]:
from petta import MeTTa, S, V

m = MeTTa().new_space()
parent = S.Parent(S.Tom, S.Bob)
person = V.person
parent, person, tuple(parent)

(Expr('(Parent Tom Bob)'),
 Var('$person'),
 (Sym('Parent'), Sym('Tom'), Sym('Bob')))

## Put a small family graph in a space

`m.run("!(+ 40 2)")` runs MeTTa source and returns one answer list for each `!` directive. Plain forms add facts to the current space. A Python-built pattern then queries those facts. Leave the `Rows` value as the last expression in a cell and Jupyter renders its columns and answers as a table.

In [2]:
m.run("""
(Parent Tom Bob)
(Parent Bob Ann)
(Parent Ann Zoe)
!(+ 40 2)
""")

[[Gnd(42)]]

In [3]:
m.query(S.Parent(V.parent, V.child))

parent,child
Tom,Bob
Bob,Ann
Ann,Zoe


## Keep Python and MeTTa in one session

Load `petta.ipython` in an ordinary Python kernel to register `%%metta`. The extension owns a default runtime, but `use(m)` points the magic at the space already created in Python. The MeTTa cell below adds Lia. The following Python query sees that fact immediately.

In [4]:
%load_ext petta.ipython
from petta.ipython import use

use(m)

In [5]:
%%metta
!(+ 1 2)
(Parent Zoe Lia)
!(match (context-space) (Parent $parent $child) ($parent $child))

3
(Tom Bob) (Bob Ann) (Ann Zoe) (Zoe Lia)


[[Gnd(3)],
 [Expr('(Tom Bob)'), Expr('(Bob Ann)'), Expr('(Ann Zoe)'), Expr('(Zoe Lia)')]]

In [6]:
m.query(S.Parent(V.parent, V.child))

parent,child
Tom,Bob
Bob,Ann
Ann,Zoe
Zoe,Lia


## Let MeTTa call a Python function

`@m.op` registers a Python callable as a MeTTa operation. Type annotations become a MeTTa declaration. Quoted MeTTa strings arrive as Python strings, and the returned string crosses back as the answer. The Python name is the MeTTa name verbatim; pass `name=` for the hyphenated spelling MeTTa prefers.

In [7]:
@m.op
def describe_link(parent_name: str, child_name: str) -> str:
    return f"{parent_name} is parent of {child_name}"

m.run('!(describe-link "Tom" "Bob")')

[[Gnd('Tom is parent of Bob')]]

## Compile a Python function into MeTTa equations

`@m.define` reads a Python function body and installs the corresponding MeTTa equation. With three parent links from Tom to Zoe, the small recursive score below returns six. `.source()` shows the installed equation, while `.py` keeps the ordinary Python function available for comparison.

In [8]:
@m.define
def generation_score(levels):
    if levels == 0:
        return 0
    return levels + generation_score(levels - 1)

generation_score.source()

'(= (generation-score $levels) (if (py-eq $levels 0) 0 (+ $levels (generation-score (- $levels 1)))))'

In [9]:
m.run("!(generation-score 3)"), generation_score.py(3)

([[Gnd(6)]], 6)

## Check a MeTTa type from Python

Declare Tom as a `Person`, then ask the engine to check that claim through `m.cast`. A successful symbolic cast returns the same symbol. A value that the requested type does not admit raises `CastError` instead of returning an approximation.

In [10]:
m.run("(: Tom Person)")
m.cast(S.Tom, "Person")

Sym('Tom')

## Read a reduction as an indented story

Trace the same recursive equation to see each call enter and each answer return. Builtins stay out of the trace, so the printed indentation follows the MeTTa function calls in the program.

In [11]:
for event in m.trace("!(generation-score 3)"):
    print(event)

-> (generation-score 3)
  -> (generation-score 2)
    -> (generation-score 1)
      -> (generation-score 0)
      (generation-score 0) = 0
    (generation-score 1) = 1
  (generation-score 2) = 3
(generation-score 3) = 6


## Find mistakes that would otherwise stay inert

MeTTa leaves many bad calls unreduced. Put two deliberate mistakes in a separate scratch space, then run `m.lint()` there. The declared function has no definition, and `one-arg` is called with two arguments. Each finding names the problem and its subject.

In [12]:
sloppy = m.new_space()
sloppy.run("""
(: ghost-fn (-> Number Number))
(= (one-arg $x) $x)
(= (bad-caller) (one-arg 1 2))
""")
for finding in sloppy.lint():
    print(finding)

[declared-but-undefined] ghost-fn: declared (-> Number Number) but nothing defines it; every call will stay unreduced
[arity-mismatch] one-arg: called with 2 argument(s) but defined for [1]


## Ask why an answer holds

Add direct and recursive ancestor equations over the same family facts. `m.derivation(...)` returns every proof of the requested answer. Leave one proof as the cell value and Jupyter renders its rules and fact leaves as a tree.

In [13]:
m.run("""
(= (ancestor $x $y)
   (match (context-space) (Parent $x $y) $y))
(= (ancestor $x $y)
   (let $middle
        (match (context-space) (Parent $x $middle0) $middle0)
        (ancestor $middle $y)))
""")
(proof,) = m.derivation(S.ancestor(S.Tom, S.Zoe))
proof

Derivation(call=Expr('(ancestor Tom Zoe)'), answer=Sym('Zoe'), children=(Step(call=Expr('(ancestor Tom Zoe)'), answer=Sym('Zoe'), equation=Expr('(= (ancestor $_157190 $_157196) (let $_157214 (match (context-space) (Parent $_157190 $_157262) $_157262) (ancestor $_157214 $_157196)))'), children=(Builtin(text="'Bob'='Bob'"), Builtin(text="'context-space'('&pyspace_1')"), Fact(space='&pyspace_1', atom=Expr('(Parent Tom Bob)')), Step(call=Expr('(ancestor Bob Zoe)'), answer=Sym('Zoe'), equation=Expr('(= (ancestor $_157770 $_157776) (let $_157794 (match (context-space) (Parent $_157770 $_157842) $_157842) (ancestor $_157794 $_157776)))'), children=(Builtin(text="'Ann'='Ann'"), Builtin(text="'context-space'('&pyspace_1')"), Fact(space='&pyspace_1', atom=Expr('(Parent Bob Ann)')), Step(call=Expr('(ancestor Ann Zoe)'), answer=Sym('Zoe'), equation=Expr('(= (ancestor $_158324 $_158330) (match (context-space) (Parent $_158324 $_158330) $_158330))'), children=(Builtin(text="'context-space'('&pyspace_1')"), Fact(space='&pyspace_1', atom=Expr('(Parent Ann Zoe)')))))))),))